In [1]:
pip install pandas numpy openpyxl xlsxwriter html5lib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import os
from io import BytesIO
import warnings
warnings.filterwarnings("ignore", "This pattern is interpreted as a regular expression")


# Set display options to show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("Libraries imported successfully")

# ================================
# CONFIGURATION
# ================================
SIGNED_HTML_PATH = r'C:\Users\Admin\Documents\GitHub\FM24-Filter_JG\File Templates\Signings.html' #Transfers
LOANS_HTML_PATH = r'C:\Users\Admin\Documents\GitHub\FM24-Filter_JG\File Templates\Loans.html' #Loans
UNIVERSAL_HTML_PATH = r'C:\Users\Admin\Documents\GitHub\FM24-Filter_JG\File Templates\Universe.html'
OUTPUT_PATH = r'C:\Users\Admin\Documents\GitHub\FM24-Filter_JG\File Templates\VALENCIA OFFICIAL V2.xlsx'
LEAGUE_MULTIPLIERS_PATH = r'C:\Users\Admin\Documents\GitHub\FM24-Filter_JG\File Templates\League Multipliers.xlsx'  # Path to your Excel file

TEXT_COLUMNS = [
    'UID', 'Name', 'Rec', 'EU National', 'Position', 'Pros',
    'Preferred Foot', 'Inf', 'Transfer Value', 'Nat', 'Division', 'Club', 'Personality'
]

PERCENTAGE_COLUMNS = ['Sv %', 'OP-Cr %', 'Hdr %', 'Conv %', 'Pas %', 'Cr C/A', 'Tck R', 'Pens Saved Ratio', 'Pen/R', 'Shot %']

# ================================
# LOAD LEAGUE POWER RATINGS FROM EXCEL
# ================================
def load_league_power(file_path):
    try:
        # Read the Excel file
        df = pd.read_excel(file_path)
        
        # Convert to dictionary with 'League' as key and 'Power Rating' as value
        league_power = dict(zip(df['League'], df['Power Rating']))
        
        # Add default 'Others' value if not present in the file
        if 'Others' not in league_power:
            league_power['Others'] = 5
            
        return league_power
    except Exception as e:
        print(f"Error loading league power ratings: {e}")
        # Return a default dictionary if file can't be loaded
        return {'Others': 5}

# Load the league power ratings
LEAGUE_POWER = load_league_power(LEAGUE_MULTIPLIERS_PATH)

# ================================
# LEAGUE NAME FIXES FOR ENCODING ISSUES
# ================================
LEAGUE_NAME_FIXES = {
    'BrasileirÃ£o AssaÃ­ SÃ©rie A': 'Brasileirão Assaí Série A',
    'Primera FederaciÃ³n Grupo I': 'Primera Federación Grupo I',
    'Liga Profesional de FÃºtbol': 'Liga Profesional de Fútbol',
    'Primera FederaciÃ³n Grupo I': 'Primera Federación Grupo',
    'Primera FederaciÃ³n Grupo III': 'Primera Federación Grupo',
    'Primera FederaciÃ³n Grupo IV': 'Primera Federación Grupo',
    'Primera FederaciÃ³n Grupo V': 'Primera Federación Grupo',
    'Primera FederaciÃ³n Grupo VI': 'Primera Federación Grupo',
    'Primera FederaciÃ³n Grupo VII': 'Primera Federación Grupo',
    'Regionalliga SÃ¼dwest': 'Regionalliga Südwest',
    'Serie C NOW Girone A': 'Serie C NOW',
    'Serie C NOW Girone B': 'Serie C NOW',
    'Serie C NOW Girone C': 'Serie C NOW',
    'Spor Toto SÃ¼per Lig': 'Spor Toto Süper Lig',
    'French National 3 - Group A': 'French National 3',
    'French National 3 - Group B': 'French National 3',
    'French National 3 - Group C': 'French National 3',
    'French National 3 - Group D': 'French National 3',
    'French National 3 - Group E': 'French National 3',
    'French National 3 - Group F': 'French National 3',
    'French National 3 - Group G': 'French National 3',
    'French National 3 - Group H': 'French National 3',
    'French National 3 - Group I': 'French National 3',
    'French National 3 - Group J': 'French National 3',
    'French National 3 - Group K': 'French National 3',
    'French National 3 - Group L': 'French National 3',
    'BrasileirÃ£o Serie B Chevrolet': 'Brasileirão Serie B Chevrolet',
    'Serie D Girone A': 'Serie D',
    'Serie D Girone B': 'Serie D',
    'Serie D Girone C': 'Serie D',
    'Serie D Girone D': 'Serie D',
    'Serie D Girone E': 'Serie D',
    'Serie D Girone F': 'Serie D',
    'Serie D Girone G': 'Serie D',
    'Serie D Girone H': 'Serie D',
    'Serie D Girone I': 'Serie D',
    'Serie D Girone J': 'Serie D',
    'Serie D Girone K': 'Serie D',
    'Regionalliga West': 'Regionalliga',
    'Regionalliga Nord': 'Regionalliga',
    'Regionalliga Südwest': 'Regionalliga',
    'Regionalliga Bayern': 'Regionalliga',
    'Regionalliga Nordost': 'Regionalliga',
    'Russian Second Division A Gold': 'Russian Second Division A',
    'Russian Second Division A Silver': 'Russian Second Division A',  
    'Russian Second Division A Bronze': 'Russian Second Division A',    
    'Russian Second Division B - Group 1': 'Russian Second Division B',
    'Russian Second Division B - Group 2': 'Russian Second Division B',
    'Russian Second Division B - Group 3': 'Russian Second Division B',
    'DR Congo Premier Division A': 'DR Congolese Premier Division',
    'DR Congo Premier Division B': 'DR Congolese Premier Division',

    # Add more fixes as needed
}

def fix_league_names(df):
    """Fix encoding issues in league names in the DataFrame"""
    if 'Division' in df.columns:
        df['Division'] = df['Division'].replace(LEAGUE_NAME_FIXES)
    return df

# ================================
# LOAD HTML FILES
# ================================
if not os.path.exists(SIGNED_HTML_PATH):
    raise FileNotFoundError(f"Signed players HTML not found: {SIGNED_HTML_PATH}")

if not os.path.exists(UNIVERSAL_HTML_PATH):
    raise FileNotFoundError(f"Universal players HTML not found: {UNIVERSAL_HTML_PATH}")

if not os.path.exists(LOANS_HTML_PATH):
    raise FileNotFoundError(f"Loan players HTML not found: {LOANS_HTML_PATH}")

with open(SIGNED_HTML_PATH, 'r', encoding='utf-8') as file:
    signed_tables = pd.read_html(file)
df_signed = signed_tables[0].copy()
df_signed = fix_league_names(df_signed)

with open(UNIVERSAL_HTML_PATH, 'r', encoding='utf-8') as file:
    universal_tables = pd.read_html(file)
df_universal = universal_tables[0].copy()
df_universal = fix_league_names(df_universal)

with open(LOANS_HTML_PATH, 'r', encoding='utf-8') as file:
    loans_tables = pd.read_html(file)
df_loans = loans_tables[0].copy()
df_loans = fix_league_names(df_loans)

#runtime 22secs

Libraries imported successfully


In [4]:
# Read the Universe.html file
try:
    universe_data = pd.read_html(UNIVERSAL_HTML_PATH)
    print(f"Successfully read HTML file")
    print(f"Number of tables found: {len(universe_data)}")
    
    # Get the first table (usually the main data)
    universe_df = universe_data[0]
    print(f"DataFrame shape: {universe_df.shape}")
    
except Exception as e:
    print(f"Error reading file: {e}")
    universe_df = None

print("Paths configured:")
print(f"Universe file: {UNIVERSAL_HTML_PATH}")

# Display all column names
if universe_df is not None:
    print("="*80)
    print("ALL COLUMN NAMES IN UNIVERSE DATASET")
    print("="*80)
    print(f"\nTotal number of columns: {len(universe_df.columns)}\n")
    
    for i, col in enumerate(universe_df.columns, 1):
        print(f"{i:3d}. {col}")
else:
    print("DataFrame not loaded. Please check the file path.")
    

Successfully read HTML file
Number of tables found: 1
DataFrame shape: (9567, 109)
Paths configured:
Universe file: C:\Users\Admin\Documents\GitHub\FM24-Filter_JG\File Templates\Universe.html
ALL COLUMN NAMES IN UNIVERSE DATASET

Total number of columns: 109

  1. Name
  2. Pref.
  3. Preferred Foot
  4. Club
  5. Yel
  6. xG
  7. Shutouts
  8. Red
  9. Pens
 10. NP-xG
 11. Conc
 12. Gls
 13. Cln/90
 14. Tck A
 15. Shot %
 16. ShT
 17. Shts Blckd
 18. Pr Passes
 19. Pres C
 20. Pres A
 21. Ps C
 22. Pas A
 23. OP-KP
 24. OP-Crs C
 25. OP-Crs A
 26. Off
 27. K Tck
 28. K Pas
 29. Itc
 30. Hdrs
 31. Goals Outside Box
 32. FK Shots
 33. xSv %
 34. xGP
 35. xG/shot
 36. Drb
 37. Dist/90
 38. Cr C
 39. Cr A
 40. Cr C/A
 41. Conv %
 42. Clr/90
 43. Clear
 44. CCC
 45. Ch C/90
 46. Blk/90
 47. Blk
 48. Inf
 49. Club.1
 50. Position
 51. Age
 52. Transfer Value
 53. Rec
 54. Aer A/90
 55. xA
 56. Asts/90
 57. UID
 58. Saves/90
 59. Tck/90
 60. Tck R
 61. Shot/90
 62. ShT/90
 63. Shots Outside 

In [5]:


# Set display options to show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("Libraries imported successfully")

Libraries imported successfully


In [6]:
# After loading each HTML file, before concatenating them
def fix_column_alignment(df):
    """Fix column alignment issues in the dataframe"""
    
    # The actual column order from your HTML (based on your sample)
    expected_columns = [
        'Club', 'Yel', 'xG', 'Tall', 'Tgls', 'Starts', 'Shutouts', 'Red', 'Pts/Gm', 'PoM',
        'Pens S', 'Pens Saved', 'Pens Faced', 'Pens', 'NP-xG', 'Last Gl', 'Last C', 'Mins/Gm',
        'Last 5 Games', 'Last 5 FT Games', 'Int Conc', 'Int Av Rat', 'Int Ast', 'Int Apps',
        'Conc', 'Gls', 'Won', 'G. Mis', 'Lost', 'D', 'Gwin', 'Cln/90', 'Av Rat', 'Mins/Gl',
        'Ast', 'Apps', 'AT Lge Gls', 'AT Lge Apps', 'AT Gls', 'Tck C', 'Tck A', 'Shot %',
        'ShT', 'Shts Blckd', 'Shots', 'Svt', 'Svp', 'Svh', 'Sv %', 'Pr Passes', 'Pres C',
        'Pres A', 'Ps C', 'Pas A', 'OP-KP', 'OP-Crs C', 'OP-Crs A', 'Off', 'K Tck', 'K Pas',
        'Itc', 'Hdrs', 'Goals Outside Box', 'FK Shots', 'xSv %', 'xGP', 'xG/shot', 'Drb',
        'Dist/90', 'Distance', 'Cr C', 'Cr A', 'Cr C/A', 'Conv %', 'Clr/90', 'Clear', 'CCC',
        'Ch C/90', 'Blk/90', 'Blk', 'Inf', 'Name', 'Club', 'Position', 'Age', 'Transfer Value',
        'Rec', 'Aer A/90', 'xA', 'Asts/90', 'UID', 'Saves/90', 'Tgls/90', 'Tcon/90', 'Tck/90',
        'Tck R', 'Shot/90', 'ShT/90', 'Shots Outside Box/90', 'Shts Blckd/90', 'Pr passes/90',
        'Pres C/90', 'Pres A/90', 'Poss Won/90', 'Poss Lost/90', 'Pen/R', 'Pens Saved Ratio',
        'Ps C/90', 'Pas %', 'Ps A/90', 'OP-KP/90', 'OP-Crs C/90', 'OP-Cr %', 'OP-Crs A/90',
        'NP-xG/90', 'Gl Mst', 'Mins', 'K Tck/90', 'K Ps/90', 'K Hdrs/90', 'Int/90', 'Sprints/90',
        'Hdr %', 'Hdrs W/90', 'Hdrs L/90', 'All/90', 'xGP/90', 'xG-OP', 'xG/90', 'xA/90',
        'Drb/90', 'FA', 'Waive Comp for Mgr Role', 'Fls', 'Gls/90', 'Cr C/90', 'Crs A/90',
        'G. Con', 'Position', 'Expires', 'EU National', 'Hdrs A', 'AT Apps', 'Division'
    ]
    
    # If we have more columns than expected, something is wrong
    if len(df.columns) > len(expected_columns):
        print(f"Warning: DataFrame has {len(df.columns)} columns, expected {len(expected_columns)}")
        # Try to identify the position column by looking at column names
        position_cols = [col for col in df.columns if 'Position' in col]
        print(f"Position columns found: {position_cols}")
        
        # If we have two position columns, the second one likely has the data
        if len(position_cols) >= 2:
            pos_col_with_data = position_cols[1]  # The second Position column
            print(f"Using {pos_col_with_data} as the source for position data")
            df['Position'] = df[pos_col_with_data]
    
    return df

# Apply this fix to each dataframe right after loading
df_signed = fix_column_alignment(df_signed)
df_universal = fix_column_alignment(df_universal)
df_loans = fix_column_alignment(df_loans)

In [7]:
# After loading all dataframes, inspect the first row to find position data
print("\n=== INSPECTING FIRST ROW OF LOANS DATA ===")
first_row = df_loans.iloc[0]
for col in df_loans.columns:
    value = first_row[col]
    if pd.notna(value) and str(value).strip() and 'AM' in str(value) or 'D ' in str(value) or 'ST' in str(value):
        print(f"Column '{col}' contains: '{value}'")


=== INSPECTING FIRST ROW OF LOANS DATA ===
Column 'Position.1' contains: 'M/AM (RLC)'


In [8]:

# ================================
# UID‑BASED MERGE AND LABEL  ✅ Fixed version
# ================================

def normalize_uid(df):
    df = df.copy()
    # Convert to float → Int64 (nullable integer) → string
    df['UID'] = pd.to_numeric(df['UID'], errors='coerce')  # force numeric, NaNs stay if bad
    df['UID'] = df['UID'].astype('Int64')                  # keep NaNs clean
    df['UID'] = df['UID'].astype(str).str.strip()          # final string
    return df

# 1️⃣ Normalize UIDs
df_signed    = normalize_uid(df_signed)
df_universal = normalize_uid(df_universal)
df_loans = normalize_uid(df_loans)

# 2️⃣ Add the signability flags
df_signed['Signability']    = 'Available for Transfer'
df_universal['Signability'] = 'Not Transferrable'
df_loans['Signability'] = 'Available on Loan'


# 3️⃣ Concatenate signables first, drop duplicate UIDs

df = (
    pd.concat([df_signed, df_loans, df_universal], ignore_index=True)
      .drop_duplicates(subset='UID', keep='first')
      .reset_index(drop=True)
)

# ================================
# ADD THIS RIGHT HERE - FIX DUPLICATE COLUMN ISSUE
# ================================
# The actual position data is in Position.1, not Position
if 'Position.1' in df.columns:
    print(f"Found Position.1 column with data. Using it as the main Position column.")
    # Replace the empty Position column with Position.1
    df['Position'] = df['Position.1']
    # Drop the duplicate column
    df = df.drop(columns=['Position.1'])
    
# Also check for other potential position columns (like Position.2 if exists)
position_cols = [col for col in df.columns if 'Position' in col]
if len(position_cols) > 1:
    print(f"Warning: Still have multiple position columns: {position_cols}")
    # If there's another Position column, check if it has data
    for col in position_cols[1:]:  # Skip the first one (which should now be our main Position)
        if df[col].notna().any():
            print(f"Column {col} has data. Using it to fill missing positions.")
            df['Position'] = df['Position'].fillna(df[col])
        df = df.drop(columns=[col])

print(f"Final position columns: {[col for col in df.columns if 'Position' in col]}")


# ================================
# FIX DUPLICATE NAME COLUMN ISSUE
# ================================
# Check for duplicate Name columns
name_cols = [col for col in df.columns if 'Name' in col]
print(f"Name columns found: {name_cols}")

if len(name_cols) > 1:
    print(f"Found multiple Name columns. Consolidating...")
    # Find which Name column actually has data
    for col in name_cols:
        non_null_count = df[col].notna().sum()
        print(f"  Column '{col}' has {non_null_count} non-null values")
    
    # Use the column with most data as the main Name
    main_name_col = max(name_cols, key=lambda col: df[col].notna().sum())
    print(f"Using '{main_name_col}' as the main Name column")
    
    # Replace the original Name column
    df['Name'] = df[main_name_col]
    
    # Drop all other Name columns except the original
    for col in name_cols:
        if col != 'Name':
            df = df.drop(columns=[col])

print(f"Final Name column: {df['Name'].notna().sum()} non-null values")
print(f"Sample names: {df['Name'].dropna().head(10).tolist()}")

# ================================
# CLEANING & CONVERSION
# ================================
def clean_and_convert_data(df):
    def convert_percentage_to_float(df, column):
        if column in df.columns:
            df[column] = (df[column].astype(str)
                          .str.replace('%', '')
                          .replace('-', np.nan)
                          .astype(float))
        return df

    # Convert minutes
    if 'Mins' in df.columns:
        df['Mins'] = pd.to_numeric(df['Mins'], errors='coerce')
        df = df[df['Mins'] >= 900].copy()

    # Convert percentage stats
    for col in PERCENTAGE_COLUMNS:
        df = convert_percentage_to_float(df, col)

    # Handle 'Dist/90' - e.g., "7.3mi"
    if 'Dist/90' in df.columns:
        df['Dist/90'] = (
            df['Dist/90']
            .astype(str)
            .str.extract(r'([\d.]+)')[0]
        )

        # Drop only if the string was completely missing
        df['Dist/90'] = pd.to_numeric(df['Dist/90'], errors='coerce')

    print("Dist/90 (after cleaning):")
    print(df['Dist/90'].describe())
    print(df['Dist/90'].dropna().head(10))






    

    # Convert all other numerical columns (except text & signability)
    for col in df.columns:
        if col not in TEXT_COLUMNS + ['Signability']:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Handle 'Dist/90' - e.g., "7.3mi"
    if 'Dist/90' in df.columns:
        df['Dist/90'] = (
            df['Dist/90']
            .astype(str)
            .str.extract(r'([\d.]+)')[0]
        )

        # Drop only if the string was completely missing
        df['Dist/90'] = pd.to_numeric(df['Dist/90'], errors='coerce')

    print("Dist/90 (after cleaning):")
    print(df['Dist/90'].describe())
    print(df['Dist/90'].dropna().head(10))


    # Add league strength multiplier
    def get_league_multiplier(division):
        return LEAGUE_POWER.get(division, LEAGUE_POWER['Others']) / 100.0

    df['League Multiplier'] = df['Division'].apply(get_league_multiplier)
    return df

# ✅ Apply cleaning
df = clean_and_convert_data(df)

# ================================
# CONVERT RAW STATS TO PER 90
# ================================
RAW_STATS_TO_PER90 = {
    "Yel": "Yellow/90",
    "Red": "Red/90",
    "Fls": "FoulsMade/90",
    "FA": "FoulsAgainst/90",
    "Off": "Offsides/90",
    "Gl Mst": "Gl Mst/90",
    "Goals Outside Box": "Goals Outside Box/90",
    "FK Shots": "FKShots/90"
}

for raw_stat, per90_stat in RAW_STATS_TO_PER90.items():
    if raw_stat in df.columns:
        df[raw_stat] = pd.to_numeric(df[raw_stat], errors='coerce')
        df['Mins'] = pd.to_numeric(df['Mins'], errors='coerce')
        df[per90_stat] = (df[raw_stat] / (df['Mins'] / 90)).fillna(0)
    else:
        print(f"Warning: Raw stat '{raw_stat}' not found in DataFrame")

print(df['Dist/90'].dropna().unique()[:50])
print(df['Dist/90'].dtype)

#runtime 1sec

Found Position.1 column with data. Using it as the main Position column.
Final position columns: ['Position']
Name columns found: ['Name']
Final Name column: 10259 non-null values
Sample names: ['Nicolas PÃ©pÃ© - Ivorian', 'RaÃºl Albiol - Spanish', 'Heorhii Sudakov - Ukrainian', 'Nahitan NÃ¡ndez - Uruguayan', 'Vitor Bueno - Brazilian', 'JoÃ£o Pedro GalvÃ£o - Italian', 'Marcelo - Brazilian', 'Divock Origi - Belgian', 'Illan Meslier - French', 'Johan Mojica - Colombian']
Dist/90 (after cleaning):
count    9805.000000
mean        4.107282
std         3.362784
min         0.000000
25%         0.000000
50%         4.600000
75%         7.400000
max         9.200000
Name: Dist/90, dtype: float64
0     0.6
1     7.2
2     3.0
3     1.2
4     2.0
5     0.0
6     3.3
7     7.5
8     3.8
10    8.8
Name: Dist/90, dtype: float64
Dist/90 (after cleaning):
count    9805.000000
mean        4.107282
std         3.362784
min         0.000000
25%         0.000000
50%         4.600000
75%         7.400000

C:\Users\Admin\AppData\Local\Temp\ipykernel_5228\3048207351.py:178: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[per90_stat] = (df[raw_stat] / (df['Mins'] / 90)).fillna(0)


In [9]:
df["Expires"] = df["Expires"].astype("string")
print(df["Expires"].dtype)

string


In [10]:
# Show overall info
print("==== DataFrame Info ====")
print(df.info())

print("\n==== Column Summary ====")
for col in df.columns:
    print(f"\n📌 Column: {col}")
    print(f"Type: {df[col].dtype}")
    print(f"Missing: {df[col].isna().sum()} / {len(df)}")
    
    if df[col].dtype == 'object':
        unique_vals = df[col].dropna().unique()
        print(f"Unique Values: {len(unique_vals)}")
        print("Sample:", unique_vals[:10])
    
    elif pd.api.types.is_numeric_dtype(df[col]):
        print(df[col].describe())

    else:
        print("Sample:", df[col].dropna().head(5).tolist())

==== DataFrame Info ====
<class 'pandas.DataFrame'>
Index: 9805 entries, 0 to 10258
Columns: 118 entries, Name to FKShots/90
dtypes: float64(104), str(13), string(1)
memory usage: 8.9 MB
None

==== Column Summary ====

📌 Column: Name
Type: str
Missing: 0 / 9805
Sample: ['Nicolas PÃ©pÃ© - Ivorian', 'RaÃºl Albiol - Spanish', 'Heorhii Sudakov - Ukrainian', 'Nahitan NÃ¡ndez - Uruguayan', 'Vitor Bueno - Brazilian']

📌 Column: Pref.
Type: float64
Missing: 8827 / 9805
count    978.000000
mean      12.046012
std       11.404579
min        1.000000
25%        5.000000
50%       10.000000
75%       15.000000
max       99.000000
Name: Pref., dtype: float64

📌 Column: Preferred Foot
Type: str
Missing: 0 / 9805
Sample: ['Left', 'Right', 'Unknown', 'Right', 'Right']

📌 Column: Club
Type: str
Missing: 0 / 9805
Sample: ['Trabzonspor', 'Villarreal', 'Shakhtar', 'Cagliari', 'ATP']

📌 Column: Yel
Type: float64
Missing: 0 / 9805
count    9805.000000
mean        1.108006
std         1.654853
min         0.

In [11]:
if df['Dist/90'].isna().any():
    print("⚠️ Warning: Missing or zero values found in Dist/90")

In [13]:

"""
# ================================
# DATA PREPROCESSING
# ================================
if 'Mins' in df.columns:
    df['Mins'] = pd.to_numeric(df['Mins'], errors='coerce')
    print(f"Minutes stats before filtering:")
    print(df['Mins'].describe())
    df = df[df['Mins'] >= 900].copy()
    print(f"After minutes filter: {len(df)} players remain")
"""

# ================================
# CREATE PER-90 COLUMNS IF THEY DON'T EXIST
# ================================
# List of raw stats that need per-90 versions
raw_to_per90 = {
    'Sprints': 'Sprints/90',
    'Int': 'Int/90',
    'K Tck': 'K Tck/90',
    'xA': 'xA/90',
    'Ch C': 'Ch C/90',
    'Gls': 'Gls/90',
    'NP-xG': 'NP-xG/90',
    'xGP': 'xGP/90',
    'Hdrs W': 'Hdrs W/90',
    'Tck': 'Tck/90',
    'OP-Crs C': 'OP-Crs C/90',
    'Drb': 'Drb/90',
    'Off': 'Offsides/90',
    'Poss Won': 'Poss Won/90',
    'Poss Lost': 'Poss Lost/90',
    'FoulsMade': 'FoulsMade/90',  # Note: you have 'Fls' not 'FoulsMade'
}

# Create missing per-90 columns
for raw_base, per90_col in raw_to_per90.items():
    if per90_col not in df.columns:
        # Try to find the raw column
        raw_col = None
        if raw_base in df.columns:
            raw_col = raw_base
        elif raw_base == 'FoulsMade' and 'Fls' in df.columns:
            raw_col = 'Fls'
        elif raw_base == 'Off' and 'Off' in df.columns:
            raw_col = 'Off'
        
        if raw_col:
            df[per90_col] = df[raw_col] / (df['Mins'] / 90)
            df[per90_col] = df[per90_col].fillna(0)
            print(f"Created {per90_col} from {raw_col}")
        else:
            df[per90_col] = 0
            print(f"⚠️ Created placeholder {per90_col} (raw column not found)")

# ================================
# CREATE COMPOSITE METRICS (WITH SAFE COLUMN ACCESS)
# ================================
# Intensity
if 'Sprints/90' in df.columns and 'Dist/90' in df.columns:
    df['Intensity'] = df["Sprints/90"] / df["Dist/90"].replace(0, 1)
else:
    df['Intensity'] = 0
    print("⚠️ Created placeholder Intensity")

# Poss_Quality
if 'Poss Won/90' in df.columns and 'Poss Lost/90' in df.columns:
    df['Poss_Quality'] = df["Poss Won/90"] / (df["Poss Lost/90"] + 0.1)
else:
    df['Poss_Quality'] = 0
    print("⚠️ Created placeholder Poss_Quality")

# Proactivity_Index
if all(col in df.columns for col in ['Int/90', 'K Tck/90', 'FoulsMade/90']):
    df['Proactivity_Index'] = (df['Int/90'] + df['K Tck/90']) / (df['FoulsMade/90'] + 1)
else:
    df['Proactivity_Index'] = 0
    print("⚠️ Created placeholder Proactivity_Index")

# Creativity_Index
if all(col in df.columns for col in ['xA/90', 'Ch C/90']):
    df['Creativity_Index'] = (df['xA/90'] * 0.7) + (df['Ch C/90'] * 0.3)
else:
    df['Creativity_Index'] = 0
    print("⚠️ Created placeholder Creativity_Index")

# Finishing_Index
if all(col in df.columns for col in ['Gls/90', 'NP-xG/90']):
    df['Finishing_Index'] = df['Gls/90'] / (df['NP-xG/90'] + 0.01)
else:
    df['Finishing_Index'] = 0
    print("⚠️ Created placeholder Finishing_Index")

print(f"\n✅ After creating metrics: {len(df)} players")



# ================================
# CHECK REQUIRED COLUMNS FOR COMPOSITE METRICS
# ================================
required_columns = ['Sprints/90', 'Dist/90', 'Poss Won/90', 'Poss Lost/90', 
                    'Int/90', 'K Tck/90', 'FoulsMade/90', 'xA/90', 'Ch C/90', 
                    'Gls/90', 'NP-xG/90']

missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    print(f"⚠️ Warning: Missing columns for composite metrics: {missing_columns}")
    print("Creating placeholder columns with zeros to avoid errors...")
    for col in missing_columns:
        df[col] = 0

# Define numeric columns to scale
TEXT_COLUMNS = ['EU National', 'Name', 'UID', 'Position', 'Personality', 'Preferred Foot', 
                'Rec', 'Club', 'Transfer Value', 'Division', 'Nat', 'Inf', 'Signability','Expires']
                
numeric_columns = [col for col in df.columns 
                   if col not in TEXT_COLUMNS 
                   and pd.api.types.is_numeric_dtype(df[col])]

# Scale numeric columns (0-1)
for col in numeric_columns:
    if col == 'Age': continue 
    if df[col].nunique() > 1:
        min_val, max_val = df[col].min(), df[col].max()
        df[col] = (df[col] - min_val) / (max_val - min_val)
    else:
        df[col] = 0.5
        
        
# ================================
# CUSTOM METRICS FOR YOUR SYSTEM
# ================================

def create_custom_metrics(df):
    """Create all custom metrics needed for Flick-style system"""
    
    df_copy = df.copy()
    
    # 1. Sweeping Index (for GK and CB)
    if all(col in df_copy.columns for col in ['Int/90', 'Dist/90', 'Sprints/90']):
        df_copy['Sweeping_Index'] = (
            0.40 * df_copy['Int/90'] +
            0.30 * df_copy['Dist/90'] +
            0.30 * df_copy['Sprints/90']
        )
    else:
        df_copy['Sweeping_Index'] = 0
        print("⚠️ Sweeping_Index created as placeholder")
    
    # 2. Passing Quality (for build-up play)
    if all(col in df_copy.columns for col in ['Pas %', 'Pr passes/90']):
        df_copy['Passing_Quality'] = (
            0.60 * df_copy['Pas %'] / 100 +  # Normalize percentage
            0.40 * df_copy['Pr passes/90']
        )
    else:
        df_copy['Passing_Quality'] = 0
        print("⚠️ Passing_Quality created as placeholder")
    
    # 3. Ball Retention (possession maintenance)
    if all(col in df_copy.columns for col in ['Poss Won/90', 'Poss Lost/90']):
        df_copy['Ball_Retention'] = df_copy['Poss Won/90'] / (df_copy['Poss Lost/90'] + 0.1)
    else:
        df_copy['Ball_Retention'] = 0
        print("⚠️ Ball_Retention created as placeholder")
    
    # 4. Defensive Pivot (for DM and CB)
    if all(col in df_copy.columns for col in ['Int/90', 'Tck/90', 'Poss Won/90']):
        df_copy['Defensive_Pivot'] = (
            0.35 * df_copy['Int/90'] +
            0.35 * df_copy['Tck/90'] +
            0.30 * df_copy['Poss Won/90']
        )
    else:
        df_copy['Defensive_Pivot'] = 0
        print("⚠️ Defensive_Pivot created as placeholder")
    
    # 5. Intensity (physical output)
    if all(col in df_copy.columns for col in ['Sprints/90', 'Dist/90']):
        df_copy['Intensity'] = df_copy['Sprints/90'] / (df_copy['Dist/90'].replace(0, 1) + 0.01)
    else:
        df_copy['Intensity'] = 0
        print("⚠️ Intensity created as placeholder")
    
    # 6. Press Resistance (ability to play under pressure)
    if 'Poss Lost/90' in df_copy.columns:
        df_copy['Press_Resistance'] = 1 / (df_copy['Poss Lost/90'] + 0.1)
    else:
        df_copy['Press_Resistance'] = 0
        print("⚠️ Press_Resistance created as placeholder")
    
    # 7. Chaos Index (for IF - cutting inside and shooting)
    if all(col in df_copy.columns for col in ['Drb/90', 'Shot/90']):
        df_copy['Chaos_Index'] = (0.50 * df_copy['Drb/90'] + 0.50 * df_copy['Shot/90'])
    else:
        df_copy['Chaos_Index'] = 0
        print("⚠️ Chaos_Index created as placeholder")
    
    # 8. Low Cross Threat (for FB - primary chance creation)
    if all(col in df_copy.columns for col in ['Cr A/90', 'xA/90']):
        df_copy['Low_Cross_Threat'] = (0.70 * df_copy['Cr A/90'] + 0.30 * df_copy['xA/90'])
    else:
        df_copy['Low_Cross_Threat'] = 0
        print("⚠️ Low_Cross_Threat created as placeholder")
    
    # 9. Pressing Output (for ST and IF - defensive work rate)
    if all(col in df_copy.columns for col in ['Sprints/90', 'Dist/90', 'Int/90']):
        df_copy['Pressing_Output'] = (
            0.50 * df_copy['Sprints/90'] +
            0.30 * df_copy['Dist/90'] +
            0.20 * df_copy['Int/90']
        )
    else:
        df_copy['Pressing_Output'] = 0
        print("⚠️ Pressing_Output created as placeholder")
    
    # 10. Progressive Value (for DM and AM - weighted progressive passes)
    if all(col in df_copy.columns for col in ['Pr passes/90', 'Pas %']):
        df_copy['Progressive_Value'] = df_copy['Pr passes/90'] * (df_copy['Pas %'] / 100)
    else:
        df_copy['Progressive_Value'] = 0
        print("⚠️ Progressive_Value created as placeholder")
    
    # 11. Chance Creation (for AM and FB)
    if all(col in df_copy.columns for col in ['xA/90', 'Ch C/90']):
        df_copy['Chance_Creation'] = (0.50 * df_copy['xA/90'] + 0.50 * df_copy['Ch C/90'])
    else:
        df_copy['Chance_Creation'] = 0
        print("⚠️ Chance_Creation created as placeholder")
    
    print("\n✅ Custom metrics created successfully")
    return df_copy


# ================================
# SAFE GET FUNCTION FOR METRIC ACCESS
# ================================

def safe_get(data, column, default=0):
    """Safely get a column value, returning default if column doesn't exist"""
    if column in data.columns:
        return data[column]
    return default

    

# ================================
# ARCHETYPE FORMULAS - FLICK-STYLE HIGH PRESS 4-2-3-1
# ================================

def get_archetype_formulas():
    """Return archetype formulas for Hansi Flick-style high-pressing 4-2-3-1 system
    
    Key Tactical Principles:
    - Gegenpress intensity: Pressing metrics prioritized across all positions
    - Low crosses as primary chance creation: Cr A/90 and xA/90 weighted heavily
    - Defense-splitting passes: Progressive passes critical for DM and AM
    - High line: Recovery pace (Sprints/90) essential for CBs and GK
    - Balanced goal threats: IF and ST both primary scorers
    """
    
    return {
        "Sweeper Keeper": {
            "filter": lambda df: df['Position'].str.contains("GK", case=False, na=False),
            "formula": lambda d: (
                0.40 * safe_get(d, "Sweeping_Index", 0) +
                0.30 * safe_get(d, "Passing_Quality", 0) +
                0.15 * safe_get(d, "Sprints/90", 0) +
                0.10 * safe_get(d, "Ball_Retention", 0) +
                0.05 * safe_get(d, "xGP/90", safe_get(d, "Sv %", 0))
            ),
            "label": "SK_Rating",
            "metrics": ["Sweeping_Index", "Passing_Quality", "Sprints/90", "Ball_Retention", "xGP/90"],
            "description": "Sweeper keeper with high line recovery, passing range, and ball retention"
        },
        
        "Central Defender": {
            "filter": lambda df: df['Position'].str.contains(r"D\s*\(C\)", regex=True, na=False),
            "formula": lambda d: (
                0.30 * safe_get(d, "Sprints/90", 0) +
                0.25 * safe_get(d, "Int/90", 0) +
                0.25 * safe_get(d, "Passing_Quality", 0) +
                0.10 * safe_get(d, "Tck/90", 0) +
                0.10 * safe_get(d, "Ball_Retention", 0)
            ),
            "label": "CD_Rating",
            "metrics": ["Sprints/90", "Int/90", "Passing_Quality", "Tck/90", "Ball_Retention"],
            "description": "High-line defender with recovery pace, interceptions, and ball-playing ability"
        },
        
        "Fullback": {
            "filter": lambda df: df['Position'].str.contains(r"(D|WB)\s*\([RL]+\)", regex=True, na=False),
            "formula": lambda d: (
                0.35 * safe_get(d, "Low_Cross_Threat", 0) +
                0.25 * safe_get(d, "Intensity", 0) +
                0.20 * safe_get(d, "Ch C/90", 0) +
                0.10 * safe_get(d, "Passing_Quality", 0) +
                0.10 * safe_get(d, "Ball_Retention", 0)
            ),
            "label": "FB_Rating",
            "metrics": ["Low_Cross_Threat", "Intensity", "Ch C/90", "Passing_Quality", "Ball_Retention"],
            "description": "Overlapping attacker specializing in low crosses, high intensity, and chance creation"
        },
        
        "Defensive Midfielder": {
            "filter": lambda df: df['Position'].str.contains(r"DM", regex=True, na=False),
            "formula": lambda d: (
                0.30 * safe_get(d, "Press_Resistance", 0) +
                0.25 * safe_get(d, "Progressive_Value", safe_get(d, "Pr passes/90", 0)) +
                0.20 * safe_get(d, "Defensive_Pivot", 0) +
                0.15 * safe_get(d, "Passing_Quality", 0) +
                0.10 * safe_get(d, "Ball_Retention", 0)
            ),
            "label": "DM_Rating",
            "metrics": ["Press_Resistance", "Progressive_Value", "Defensive_Pivot", "Passing_Quality", "Ball_Retention"],
            "description": "Pure anchor with press resistance and defense-splitting progressive passes"
        },
        
        "Attacking Midfielder": {
            "filter": lambda df: df['Position'].str.contains(r"AM\s*\(C\)", regex=True, na=False),
            "formula": lambda d: (
                0.35 * safe_get(d, "Ch C/90", 0) +
                0.25 * safe_get(d, "Progressive_Value", safe_get(d, "Pr passes/90", 0)) +
                0.20 * safe_get(d, "NP-xG/90", 0) +
                0.10 * safe_get(d, "Passing_Quality", 0) +
                0.10 * safe_get(d, "Ball_Retention", 0)
            ),
            "label": "AM_Rating",
            "metrics": ["Ch C/90", "Progressive_Value", "NP-xG/90", "Passing_Quality", "Ball_Retention"],
            "description": "Creative playmaker with defense-splitting passes, chance creation, and goal threat"
        },
        
        "Inside Forward": {
            "filter": lambda df: df['Position'].str.contains(r"AM\s*\([RL]+\)", regex=True, na=False),
            "formula": lambda d: (
                0.30 * safe_get(d, "NP-xG/90", 0) +
                0.25 * safe_get(d, "Chaos_Index", 0) +
                0.20 * safe_get(d, "Pressing_Output", 0) +
                0.15 * safe_get(d, "Ch C/90", 0) +
                0.10 * safe_get(d, "Ball_Retention", 0)
            ),
            "label": "IF_Rating",
            "metrics": ["NP-xG/90", "Chaos_Index", "Pressing_Output", "Ch C/90", "Ball_Retention"],
            "description": "Goal-scoring winger who cuts inside, presses aggressively, and creates chaos"
        },
        
        "Striker": {
            "filter": lambda df: df['Position'].str.contains(r"ST", regex=True, na=False),
            "formula": lambda d: (
                0.30 * safe_get(d, "Pressing_Output", 0) +
                0.25 * safe_get(d, "NP-xG/90", 0) +
                0.20 * safe_get(d, "Gls/90", safe_get(d, "NP-xG/90", 0)) +
                0.15 * safe_get(d, "Hdrs W/90", 0) +
                0.10 * safe_get(d, "Ball_Retention", 0)
            ),
            "label": "ST_Rating",
            "metrics": ["Pressing_Output", "NP-xG/90", "Gls/90", "Hdrs W/90", "Ball_Retention"],
            "description": "Pressing forward who triggers counter-press, scores goals, and wins aerial duels"
        }
    }
    
# ADD THIS LINE - CALL THE FUNCTION TO CREATE THE DICTIONARY
ARCHETYPE_FORMULAS = get_archetype_formulas()

# ================================
# DEBUG: Check if any data will be written
# ================================
print(f"\n📊 DataFrame shape: {df.shape}")
print(f"Columns available: {list(df.columns)}")
print(f"Sample of first 5 rows:")
print(df.head(5)[['UID', 'Name', 'Position'] if all(col in df.columns for col in ['UID', 'Name', 'Position']) else df.columns[:5]])

# Check if any archetype filters will produce results
for role, config in ARCHETYPE_FORMULAS.items():
    if 'filter' in config:
        filtered_count = config['filter'].sum() if hasattr(config['filter'], 'sum') else 0
        print(f"  {role}: {filtered_count} players match filter")

# ================================
# OUTPUT TO EXCEL WITH SUMMARY SHEET
# ================================
output_excel = BytesIO()

with pd.ExcelWriter(output_excel, engine='xlsxwriter', engine_kwargs={'options': {'strings_to_urls': False}}) as writer:
    workbook = writer.book
    unicode_format = workbook.add_format({'font_name': 'Arial Unicode MS', 'valign': 'vcenter'})
    
    # First create the summary sheet at the beginning
    worksheet = workbook.add_worksheet('Player Archetype Summary')
    writer.sheets['Player Archetype Summary'] = worksheet
    
    # Dictionary to store archetype data for summary
    summary_data = []
    
    # Process all archetypes
    for role, config in ARCHETYPE_FORMULAS.items():
        role_df = df[config["filter"]].copy()
        if role_df.empty:
            continue

        # Calculate ratings
        role_df[config["label"]] = config["formula"](role_df)
        adjusted_label = f"Adjusted {config['label']}"
        role_df[adjusted_label] = role_df[config["label"]] * role_df['League Multiplier']
        
        # Calculate percentile for each player within this archetype
        role_df['Percentile'] = role_df[adjusted_label].rank(pct=True)
        
        # Store data for summary
        top_players = role_df[role_df['Percentile'] > 0.95][['UID', 'Name', 'Percentile']].copy()
        top_players['Archetype'] = role
        summary_data.append(top_players)
        
        # Write archetype sheet
        role_df['Ranking'] = role_df[adjusted_label].rank(method='min', ascending=False)
        
        output_cols = [
            'UID', 'Name', 'Age', 'Personality', 'Signability', 'EU National', 'Position', 'Preferred Foot',
            'Transfer Value', 'Nat', 'Division', 'Club',
            config["label"], adjusted_label, 'Percentile', 'Ranking', 'League Multiplier', 'Expires'
        ]
        
        result_df = role_df[output_cols].copy().sort_values(by=adjusted_label, ascending=False)
        result_df.to_excel(writer, sheet_name=role, index=False)
        
        # Format archetype sheet
        archetype_sheet = writer.sheets[role]
        for i, col in enumerate(output_cols):
            archetype_sheet.set_column(i, i, 20 if col != 'Name' else 30, unicode_format)
        
        # Format percentile as percentage
        if 'Percentile' in output_cols:
            percent_format = workbook.add_format({'num_format': '0.0%'})
            percentile_col_idx = output_cols.index('Percentile')
            archetype_sheet.set_column(percentile_col_idx, percentile_col_idx, 12, percent_format)
    
    # ================================
    # BUILD SUMMARY SHEET DATA
    # ================================
    if summary_data:
        all_archetypes = pd.concat(summary_data).sort_values(['UID', 'Percentile'], ascending=[True, False])
        
        # Group and format archetypes
        def format_archetypes(group):
            return ", ".join(f"{row['Archetype']} ({row['Percentile']:.1%})" for _, row in group.iterrows())
        
        player_summary = (
            all_archetypes.groupby(['UID', 'Name'])
            .apply(format_archetypes)
            .reset_index(name='Top Archetypes (>95%)')
        )
        
        # Add additional info
        player_summary = player_summary.merge(
            df[['UID', 'Position', 'Age', 'Nat', 'Club', 'Division', 'Personality', 'Signability', 'Transfer Value']],
            on='UID', how='left'
        )
        
        # Final columns and sorting
        player_summary = player_summary[[
            'UID', 'Name', 'Position', 'Club', 'Division', 
            'Signability', 'Transfer Value', 'Age', 'Nat', 'Personality', 'Top Archetypes (>95%)'
        ]]
        player_summary['Archetype Count'] = player_summary['Top Archetypes (>95%)'].str.count(',') + 1
        player_summary = player_summary.sort_values(['Archetype Count', 'Name'], ascending=[False, True])
        
        # Write to summary sheet
        player_summary.to_excel(writer, sheet_name='Player Archetype Summary', index=False)
        
        # Format summary sheet
        summary_sheet = writer.sheets['Player Archetype Summary']
        summary_sheet.set_column('A:A', 15)  # UID
        summary_sheet.set_column('B:B', 30)  # Name
        summary_sheet.set_column('C:C', 15)  # Position
        summary_sheet.set_column('D:D', 25)  # Club
        summary_sheet.set_column('E:E', 25)  # Division
        summary_sheet.set_column('F:F', 15)  # Signability
        summary_sheet.set_column('G:G', 15)  # Transfer Value
        summary_sheet.set_column('H:H', 60)  # Age
        summary_sheet.set_column('I:I', 60)  # Nationality
        summary_sheet.set_column('J:J', 60)  # Personality
        summary_sheet.set_column('K:K', 60)  # Top Archetypes

# ================================
# SAVE FILE
# ================================
try:
    with open(OUTPUT_PATH, 'wb') as output_file:
        output_file.write(output_excel.getbuffer())
    print(f"\n✅ Success! Output file saved:\n{OUTPUT_PATH}")
    print(f"Includes {len(ARCHETYPE_FORMULAS)} archetype sheets + summary sheet")
except Exception as e:
    print(f"\n❌ Error saving file: {e}")
    print("Please check directory permissions or disk space.")


✅ After creating metrics: 9805 players

📊 DataFrame shape: (9805, 123)
Columns available: ['Name', 'Pref.', 'Preferred Foot', 'Club', 'Yel', 'xG', 'Shutouts', 'Red', 'Pens', 'NP-xG', 'Conc', 'Gls', 'Cln/90', 'Tck A', 'Shot %', 'ShT', 'Shts Blckd', 'Pr Passes', 'Pres C', 'Pres A', 'Ps C', 'Pas A', 'OP-KP', 'OP-Crs C', 'OP-Crs A', 'Off', 'K Tck', 'K Pas', 'Itc', 'Hdrs', 'Goals Outside Box', 'FK Shots', 'xSv %', 'xGP', 'xG/shot', 'Drb', 'Dist/90', 'Cr C', 'Cr A', 'Cr C/A', 'Conv %', 'Clr/90', 'Clear', 'CCC', 'Ch C/90', 'Blk/90', 'Blk', 'Inf', 'Club.1', 'Position', 'Age', 'Transfer Value', 'Rec', 'Aer A/90', 'xA', 'Asts/90', 'UID', 'Saves/90', 'Tck/90', 'Tck R', 'Shot/90', 'ShT/90', 'Shots Outside Box/90', 'Shts Blckd/90', 'Pr passes/90', 'Pres C/90', 'Pres A/90', 'Poss Won/90', 'Poss Lost/90', 'Pen/R', 'Pens Saved Ratio', 'Ps C/90', 'Pas %', 'Ps A/90', 'OP-KP/90', 'OP-Crs C/90', 'OP-Cr %', 'OP-Crs A/90', 'NP-xG/90', 'Gl Mst', 'Mins', 'K Tck/90', 'K Ps/90', 'K Hdrs/90', 'Int/90', 'Sprints

In [15]:
# ================================
# COMPREHENSIVE COLUMN INSPECTION
# ================================
print("\n" + "="*80)
print("COLUMN INSPECTION")
print("="*80)

# Get first row of data
first_row = df.iloc[0]

# Print all columns with their first value
print("\n📋 ALL COLUMNS WITH SAMPLE VALUES (first row):")
print("-" * 80)
for col in df.columns:
    value = first_row[col]
    value_str = str(value) if pd.notna(value) else "NaN"
    # Truncate long values
    if len(value_str) > 50:
        value_str = value_str[:50] + "..."
    print(f"{col:30} : {value_str}")

# Find columns that might contain text/names
print("\n" + "="*80)
print("🔍 SEARCHING FOR COLUMNS WITH TEXT DATA")
print("="*80)

text_columns = []
for col in df.columns:
    if df[col].dtype == 'object':  # Text columns
        # Get sample of non-null values
        sample = df[col].dropna().head(20).tolist()
        if sample:
            # Filter out numeric-looking strings and placeholders
            text_values = [str(v) for v in sample 
                          if isinstance(v, str) 
                          and len(str(v)) > 1 
                          and not str(v).replace('.', '').replace('-', '').strip().isdigit()
                          and str(v) not in ['-', '--', '- -', 'NaN', 'nan', '']]
            
            if text_values:
                unique_text = list(set(text_values))[:5]
                print(f"\n📌 Column: '{col}'")
                print(f"   Total non-null: {df[col].notna().sum()}")
                print(f"   Sample values: {unique_text}")
                text_columns.append(col)

# Check specifically for player names
print("\n" + "="*80)
print("👤 LOOKING FOR PLAYER NAMES")
print("="*80)

name_patterns = ['Name', 'PLAYER', 'Player', 'NAME', 'name', 'Nom', 'Prenom', 'Surname']
for pattern in name_patterns:
    matching_cols = [col for col in df.columns if pattern in col]
    if matching_cols:
        for col in matching_cols:
            sample = df[col].dropna().head(10).tolist()
            print(f"\nColumn '{col}' (matches '{pattern}'):")
            print(f"  Sample: {sample}")

# Show value counts for potential text columns
print("\n" + "="*80)
print("📊 VALUE COUNTS FOR TEXT COLUMNS")
print("="*80)

for col in text_columns[:10]:  # Limit to first 10 text columns
    print(f"\n{col} (top 10 values):")
    print(df[col].value_counts().head(10))

# Check if UID can be used as identifier
print("\n" + "="*80)
print("🔑 UID COLUMN INFO")
print("="*80)
if 'UID' in df.columns:
    print(f"UID - unique count: {df['UID'].nunique()}")
    print(f"UID - sample: {df['UID'].dropna().head(10).tolist()}")
else:
    print("No UID column found!")

print("\n" + "="*80)
print("INSPECTION COMPLETE")
print("="*80)


COLUMN INSPECTION

📋 ALL COLUMNS WITH SAMPLE VALUES (first row):
--------------------------------------------------------------------------------
Name                           : Nicolas PÃ©pÃ© - Ivorian
Pref.                          : NaN
Preferred Foot                 : Left
Club                           : Trabzonspor
Yel                            : 0.0
xG                             : 0.2533191715347849
Shutouts                       : NaN
Red                            : 0.0
Pens                           : 0.0
NP-xG                          : 0.28477611940298503
Conc                           : NaN
Gls                            : 0.034482758620689655
Cln/90                         : NaN
Tck A                          : 0.15384615384615385
Shot %                         : 0.42
ShT                            : 0.19480519480519481
Shts Blckd                     : 0.0
Pr Passes                      : 0.01834862385321101
Pres C                         : 0.07526881720430108
Pres A 

In [16]:
# ================================
# LIST ALL UNUSED COLUMNS
# ================================
print("\n" + "="*80)
print("📊 UNUSED COLUMNS ANALYSIS")
print("="*80)

# Get all columns in the dataframe
all_columns = set(df.columns.tolist())
print(f"Total columns in dataframe: {len(all_columns)}")

# Define all columns that ARE used in your code
used_columns = set()

# 1. Columns used in TEXT_COLUMNS
used_columns.update(TEXT_COLUMNS)
used_columns.update(['Display_Name'])  # If you added this

# 2. Columns used in PERCENTAGE_COLUMNS
used_columns.update(PERCENTAGE_COLUMNS)

# 3. Columns used in RAW_STATS_TO_PER90 (both raw and per90)
for raw_stat, per90_stat in RAW_STATS_TO_PER90.items():
    used_columns.add(raw_stat)
    used_columns.add(per90_stat)

# 4. Columns used in raw_to_per90 mapping
for raw_base, per90_col in raw_to_per90.items():
    used_columns.add(per90_col)
    # Add possible raw column names
    if raw_base in df.columns:
        used_columns.add(raw_base)
    elif raw_base == 'FoulsMade' and 'Fls' in df.columns:
        used_columns.add('Fls')
    elif raw_base == 'Off' and 'Off' in df.columns:
        used_columns.add('Off')

# 5. Columns used in composite metrics
composite_metrics = ['Intensity', 'Poss_Quality', 'Proactivity_Index', 
                     'Creativity_Index', 'Finishing_Index']
used_columns.update(composite_metrics)

# 6. Columns used in archetype formulas
archetype_columns = [
    'xGP/90', 'Sv %', 'Pas %', 'Gl Mst/90', 'Hdrs W/90', 'Tck/90', 
    'OP-Crs C/90', 'xA/90', 'Int/90', 'Ps %', 'Tck R', 'NP-xG/90', 
    'Drb/90', 'Ch C/90', 'Pr passes/90', 'xG/shot', 'Offsides/90',
    'League Multiplier', 'Mins', 'Dist/90', 'Sprints/90', 'Poss Won/90',
    'Poss Lost/90', 'K Tck/90', 'FoulsMade/90', 'Gls/90'
]
used_columns.update(archetype_columns)

# 7. Columns used in filtering and output
filter_columns = ['Position', 'Mins', 'Age']
used_columns.update(filter_columns)

# 8. Columns that are essential for the output structure
essential_columns = ['UID', 'Name', 'Club', 'Division', 'Signability', 
                     'EU National', 'Preferred Foot', 'Transfer Value', 
                     'Nat', 'Personality', 'Expires', 'Rec', 'Inf']
used_columns.update(essential_columns)

# 9. League Multiplier column
used_columns.add('League Multiplier')

# 10. Per-90 columns created from raw stats
per90_created = ['Yellow/90', 'Red/90', 'FoulsMade/90', 'FoulsAgainst/90', 
                 'Offsides/90', 'Gl Mst/90', 'Goals Outside Box/90', 'FKShots/90']
used_columns.update(per90_created)

# Filter to only include columns that actually exist in the dataframe
used_columns = {col for col in used_columns if col in df.columns}

# Find unused columns
unused_columns = all_columns - used_columns

print(f"\n✅ Used columns: {len(used_columns)}")
print(f"❌ Unused columns: {len(unused_columns)}")

# Show unused columns grouped by category
print("\n" + "="*80)
print("📋 UNUSED COLUMNS BY CATEGORY")
print("="*80)

# Categorize unused columns
def categorize_column(col):
    col_lower = col.lower()
    if 'per90' in col_lower or '/90' in col_lower or 'per 90' in col_lower:
        return "Per-90 Stats"
    elif any(x in col_lower for x in ['tck', 'tackl', 'def', 'header', 'hdr', 'blk', 'clr', 'int']):
        return "Defensive Stats"
    elif any(x in col_lower for x in ['shot', 'goal', 'gls', 'xG', 'np-xg', 'finish']):
        return "Attacking Stats"
    elif any(x in col_lower for x in ['pass', 'ps ', 'pr ', 'kp', 'cross', 'crs', 'ch c', 'xa']):
        return "Passing/Creativity"
    elif any(x in col_lower for x in ['sprint', 'dist', 'intensity', 'pace']):
        return "Physical Stats"
    elif any(x in col_lower for x in ['yel', 'red', 'fls', 'fa', 'off', 'foul']):
        return "Discipline Stats"
    elif any(x in col_lower for x in ['gk', 'sv', 'save', 'xgp', 'pens saved']):
        return "Goalkeeper Stats"
    elif any(x in col_lower for x in ['age', 'mins', 'apps', 'starts']):
        return "Playing Time"
    elif any(x in col_lower for x in ['club', 'div', 'league', 'nation', 'nat']):
        return "Context Info"
    else:
        return "Other"

# Group unused columns by category
unused_by_category = {}
for col in sorted(unused_columns):
    category = categorize_column(col)
    if category not in unused_by_category:
        unused_by_category[category] = []
    unused_by_category[category].append(col)

# Print unused columns by category
for category, cols in unused_by_category.items():
    print(f"\n📌 {category} ({len(cols)} columns):")
    # Print in rows of 5 for readability
    for i in range(0, len(cols), 5):
        row_cols = cols[i:i+5]
        print(f"   {', '.join(row_cols)}")

# Summary statistics
print("\n" + "="*80)
print("📊 SUMMARY STATISTICS")
print("="*80)
print(f"Total columns: {len(all_columns)}")
print(f"Columns used in analysis: {len(used_columns)}")
print(f"Columns not used: {len(unused_columns)}")
print(f"Usage rate: {len(used_columns)/len(all_columns)*100:.1f}%")

# Optional: Show sample of used columns
print("\n✅ SAMPLE OF USED COLUMNS (first 20):")
used_list = sorted(list(used_columns))[:20]
for i, col in enumerate(used_list):
    print(f"  {i+1:2}. {col}")

print("\n" + "="*80)


📊 UNUSED COLUMNS ANALYSIS
Total columns in dataframe: 123

✅ Used columns: 72
❌ Unused columns: 51

📋 UNUSED COLUMNS BY CATEGORY

📌 Per-90 Stats (23 columns):
   Aer A/90, All/90, Asts/90, Blk/90, Cln/90
   Clr/90, Cr C/90, Crs A/90, Hdrs L/90, K Hdrs/90
   K Ps/90, OP-Crs A/90, OP-KP/90, Pres A/90, Pres C/90
   Ps A/90, Ps C/90, Saves/90, ShT/90, Shot/90
   Shots Outside Box/90, Shts Blckd/90, xG/90

📌 Defensive Stats (4 columns):
   Blk, Hdrs, Hdrs A, Tck A

📌 Other (18 columns):
   CCC, Clear, Conc, Cr A, Cr C
   G. Con, Itc, K Pas, Pas A, Pens
   Pref., Pres A, Pres C, ShT, Shts Blckd
   Shutouts, xG, xG-OP

📌 Context Info (1 columns):
   Club.1

📌 Passing/Creativity (4 columns):
   OP-Crs A, OP-KP, Pr Passes, Ps C

📌 Goalkeeper Stats (1 columns):
   xSv %

📊 SUMMARY STATISTICS
Total columns: 123
Columns used in analysis: 72
Columns not used: 51
Usage rate: 58.5%

✅ SAMPLE OF USED COLUMNS (first 20):
   1. Age
   2. Ch C/90
   3. Club
   4. Conv %
   5. Cr C/A
   6. Creativity_Ind